# Cajamarca · Clasificación del índice estimado E1–E4
**Prototipo académico · CRISP-DM · Google Colab**

Este cuaderno compara Árbol de Decisión, Random Forest, XGBoost y SVM para anticipar
la banda mensual de un índice de precipitación y susceptibilidad en los 12 distritos.
La referencia son **etiquetas calculadas E1–E4**. Su desempeño no acredita peligro
físico observado ni niveles oficiales de riesgo.

**Para empezar:** sube este `.ipynb` a Colab mediante **Archivo → Subir cuaderno**,
elige un entorno **CPU** y ejecuta **Entorno de ejecución → Ejecutar todas**.
Cuando aparezca el selector, carga **datos_cajamarca.zip** sin descomprimir.
Mantén la sesión activa hasta la exportación. El mapa y sus controles funcionan
cuando las celdas están ejecutadas y la sesión sigue conectada.

Los resultados se calculan al ejecutar; este archivo no contiene métricas inventadas.
No cambies los cortes, entradas ni particiones después de observar prueba.


## 1 · Alcance y contrato
Unidad: distrito-mes. Horizonte: mes objetivo **t**, utilizando lluvia hasta **t−1**.
Población documental: 6.480 combinaciones; muestra: 6.408, tras excluir seis meses
iniciales por distrito. Son observaciones repetidas de 12 territorios.

Cinco entradas: mes, susceptibilidad fija, lluvia del mes anterior y acumulados
previos de tres y seis meses. La lluvia realizada de t define la referencia y se
excluye de X. No se incorporan pobreza, población, daños ni INDECI al estimador.

Limitaciones: calendario nominal reconstruido sin confirmación nativa; referencia
territorial fija sin disponibilidad histórica acreditada. Se trata de un experimento
retrospectivo, sin demostrar disponibilidad de los productos en tiempo real.


## 2 · Entorno reproducible
Se fijan versiones de las herramientas de modelado y visualización; se registran
también las versiones de NumPy y Pandas proporcionadas por el entorno.
Ejecutar esta celda antes de importar bibliotecas. CPU es suficiente.


In [ ]:
import sys, subprocess, importlib.metadata as metadata
PAQUETES = ['scikit-learn==1.7.2', 'xgboost==3.1.1', 'matplotlib==3.10.7',
            'seaborn==0.13.2', 'folium==0.20.0', 'ipywidgets==8.1.7', 'joblib==1.5.2']
pendientes = []
for paquete in PAQUETES:
    nombre, version = paquete.split('==')
    try:
        coincide = metadata.version(nombre) == version
    except metadata.PackageNotFoundError:
        coincide = False
    if not coincide:
        pendientes.append(paquete)
if pendientes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *pendientes])
print('Bibliotecas de modelado preparadas.')


In [ ]:
import os, sys, json, hashlib, platform, random, time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
BASE = Path('/content/cajamarca') if EN_COLAB else Path.cwd()
DATA_DIR = BASE / 'datos'
OUT = BASE / 'resultados'
OUT.mkdir(parents=True, exist_ok=True)
VERSIONES = {p: metadata.version(p) for p in
             ['numpy','pandas','scikit-learn','xgboost','matplotlib','seaborn','folium','ipywidgets','joblib']}
print('Python:', platform.python_version(), '| entorno:', 'Colab' if EN_COLAB else 'local')
display(pd.Series(VERSIONES, name='Versión').to_frame())


## 3 · Carga de archivos y procedencia
Se verifica la identidad de los archivos mediante SHA-256 antes de leerlos. La
extracción acepta únicamente los nombres previstos, sin extraer rutas arbitrarias.
Los CSV usan punto y coma, coma decimal y UTF-8 con BOM; UBIGEO se conserva como texto.
El archivo completo sirve para auditoría: sólo las cinco columnas declaradas pasan a X.


In [ ]:
ESPERADOS = {'entrenamiento_proxy_6408.csv': '8c38caa1ec7a11999d76916825589aa8aa8531bd4c2bd9223b12ff4ce306b0b7', 'dataset_cajamarca_6480_etiquetas_estimadas.csv': '314a70aeb0f4f9ff4b96647bc9d84c82079506162c70d6e74b5e9687c82beb2b', 'columnas_para_modelo.json': 'c5e5ea47c41d2a5bc65c63b8dde26ec27b2a6cbac3cd23eb14c1af6faba22b88', 'LEEME_METODO.md': '3fbc5967a555e4865da8230f3f9f40cfe923c1a579a56a384f409b19d2f53ff1', 'distritos_cajamarca_mascara_poligonal.geojson': '1dbfb00d3b4a0cc8287db8eaff71978c59e3af0340053908ab319049fe144ac1'}
import zipfile, io
if not all((DATA_DIR / nombre).exists() for nombre in ESPERADOS):
    if not EN_COLAB:
        raise FileNotFoundError('Coloca la carpeta datos junto al cuaderno, o ejecuta en Colab y sube datos_cajamarca.zip.')
    from google.colab import files
    print('Selecciona datos_cajamarca.zip')
    subidos = files.upload()
    candidatos = [contenido for nombre, contenido in subidos.items() if nombre.lower().endswith('.zip')]
    if len(candidatos) != 1:
        raise ValueError('Carga exactamente un archivo ZIP de datos.')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(candidatos[0])) as archivo:
        for nombre, huella in ESPERADOS.items():
            info = archivo.getinfo(nombre)
            if info.file_size > 20_000_000:
                raise ValueError('Tamaño inesperado en '+nombre)
            contenido = archivo.read(nombre)
            if hashlib.sha256(contenido).hexdigest() != huella:
                raise ValueError('El contenido no coincide con la versión preparada: '+nombre)
            (DATA_DIR / nombre).write_bytes(contenido)
for nombre, huella in ESPERADOS.items():
    if hashlib.sha256((DATA_DIR / nombre).read_bytes()).hexdigest() != huella:
        raise ValueError('Archivo modificado: '+nombre)
config = json.loads((DATA_DIR / 'columnas_para_modelo.json').read_text(encoding='utf-8-sig'))
FEATURES = config['entradas']
CLASES = config['etiquetas']
TARGET = config['objetivo']
def leer_csv(nombre):
    tabla = pd.read_csv(DATA_DIR / nombre, sep=';', decimal=',', encoding='utf-8-sig', dtype={'ubigeo':'string'})
    tabla['periodo'] = pd.to_datetime(tabla['periodo'], format='%Y-%m', errors='raise')
    return tabla.sort_values(['periodo','ubigeo']).reset_index(drop=True)
df = leer_csv('entrenamiento_proxy_6408.csv')
completo = leer_csv('dataset_cajamarca_6480_etiquetas_estimadas.csv')
geojson = json.loads((DATA_DIR / 'distritos_cajamarca_mascara_poligonal.geojson').read_text(encoding='utf-8-sig'))
print('Modelado:', df.shape, '| panel completo:', completo.shape)
display(df.head(3))
print((DATA_DIR / 'LEEME_METODO.md').read_text(encoding='utf-8-sig'))


## 4 · Calidad y cobertura
La auditoría se detiene ante claves duplicadas, coberturas incorrectas, números
inválidos, antecedentes inconsistentes o etiquetas que no se reproducen.


In [ ]:
"""Celda de auditoría del panel preparado; no entrena ni modifica los originales."""
import numpy as np
import pandas as pd
from IPython.display import display


def validar_datos(df, completo, config, geojson, FEATURES, CLASES):
    """Detener el cuaderno si los datos incumplen el protocolo E1–E4 fijado."""
    def exigir(condicion, mensaje):
        if not bool(condicion):
            raise ValueError("Auditoría detenida: " + mensaje)

    entradas = ["mes", "susceptibilidad_mm_cenepred", "precip_mes_previo_mm",
                "precip_acum_3meses_previos_mm", "precip_acum_6meses_previos_mm"]
    bandas = ["E1", "E2", "E3", "E4"]
    claves = ["ubigeo", "periodo"]
    objetivo = "banda_proxy_estimada"
    ubigeos = {f"0601{i:02d}" for i in range(1, 13)}
    cortes = np.array([0.25, 0.50, 0.75], dtype=np.float64)
    exigir(list(FEATURES) == entradas == config.get("entradas"), "las cinco entradas o su orden cambiaron.")
    exigir(list(CLASES) == bandas == config.get("etiquetas"), "el diccionario E1–E4 cambió.")
    exigir(config.get("objetivo") == objetivo, "la variable objetivo no coincide con la configuración.")
    exigir(config.get("particion") == "particion_temporal", "la partición configurada cambió.")
    exigir(config.get("dtype", {}).get("ubigeo") == "string", "UBIGEO debe leerse como texto.")
    parametros = config["parametros"]
    exigir(parametros.get("cortes_oficiales") is False and parametros.get("probabilidad") is False,
            "el índice no representa categorías oficiales ni probabilidades.")
    exigir(parametros.get("referencia") == ["1981-01", "2002-12"], "el periodo de referencia cambió.")
    exigir(parametros.get("observaciones_referencia") == 3168, "la referencia debe contener 3 168 observaciones.")
    cortes_documentados = {"E1": "0 <= I < 0.25", "E2": "0.25 <= I < 0.50",
                           "E3": "0.50 <= I < 0.75", "E4": "I >= 0.75"}
    exigir(parametros.get("cortes") == cortes_documentados, "los cortes E1–E4 cambiaron.")

    metadatos = [objetivo, "particion_temporal", "etiqueta_estimada", "origen_etiqueta",
                 "estado_cronologia", "atributos_cenepred_sin_validez_historica_confirmada"]
    requeridas = set(claves + ["distrito"] + entradas + metadatos)
    extra_completo = {"precipitacion_mm", "indice_proxy_estimado", "apto_modelo_proxy"}
    exigir(requeridas.issubset(df.columns), "faltan columnas en el archivo de modelado.")
    exigir((requeridas | extra_completo).issubset(completo.columns), "faltan columnas en el panel completo.")
    exigir(len(df) == 6408 and len(completo) == 6480, "se esperaban 6 408 y 6 480 filas.")
    exigir(not df.columns.duplicated().any() and not completo.columns.duplicated().any(), "hay columnas duplicadas.")

    panel = completo.copy().sort_values(claves).reset_index(drop=True)
    modelo = df.copy().sort_values(claves).reset_index(drop=True)
    meses_full = pd.date_range("1981-01-01", "2025-12-01", freq="MS")
    meses_modelo = pd.date_range("1981-07-01", "2025-12-01", freq="MS")
    for nombre, tabla, meses in [("panel completo", panel, meses_full),
                                  ("modelado", modelo, meses_modelo)]:
        exigir(pd.api.types.is_datetime64_any_dtype(tabla["periodo"]),
                f"periodo debe convertirse a fecha mensual antes de auditar {nombre}.")
        exigir(tabla["periodo"].notna().all(), f"hay periodos vacíos en {nombre}.")
        exigir((tabla["periodo"].dt.day == 1).all(), f"los periodos de {nombre} no son inicios de mes.")
        exigir(tabla[claves].notna().all().all(), f"faltan claves en {nombre}.")
        exigir(not tabla.duplicated(claves).any(), f"hay claves distrito-mes duplicadas en {nombre}.")
        exigir(tabla["ubigeo"].map(lambda x: isinstance(x, str) and len(x) == 6).all(),
                f"UBIGEO perdió sus ceros iniciales en {nombre}.")
        exigir(set(tabla["ubigeo"]) == ubigeos, f"no están los 12 UBIGEO previstos en {nombre}.")
        esperadas = pd.MultiIndex.from_product([sorted(ubigeos), meses], names=claves)
        reales = pd.MultiIndex.from_frame(tabla[claves])
        exigir(len(esperadas.difference(reales)) == 0 and len(reales.difference(esperadas)) == 0,
                f"faltan o sobran combinaciones distrito-mes en {nombre}.")
        exigir(tabla.groupby("periodo")["ubigeo"].nunique().eq(12).all(),
                f"hay meses sin los 12 distritos en {nombre}.")
        exigir(tabla.groupby("ubigeo")["distrito"].nunique(dropna=False).eq(1).all()
                and tabla["distrito"].notna().all(), f"nombres distritales inconsistentes en {nombre}.")
        mes = pd.to_numeric(tabla["mes"], errors="raise").to_numpy(dtype=float)
        exigir(np.array_equal(mes, tabla["periodo"].dt.month.to_numpy()), f"mes no coincide con periodo en {nombre}.")
        exigir(tabla[objetivo].notna().all() and set(tabla[objetivo]) == set(bandas),
                f"hay etiquetas ajenas a E1–E4 o faltantes en {nombre}.")
        exigir(pd.to_numeric(tabla["etiqueta_estimada"], errors="raise").eq(1).all(),
                f"se alteró la identificación de etiquetas calculadas en {nombre}.")
        exigir(tabla["origen_etiqueta"].eq("regla_exploratoria_lluvia_susceptibilidad_v1").all(),
                f"el origen de las etiquetas cambió en {nombre}.")
        exigir(tabla["estado_cronologia"].eq("orden_cronologico_asumido_no_fecha_nativa").all(),
                f"la condición del calendario reconstruido cambió en {nombre}.")
        exigir(pd.to_numeric(tabla["atributos_cenepred_sin_validez_historica_confirmada"], errors="raise").eq(1).all(),
                f"la advertencia de vigencia de susceptibilidad cambió en {nombre}.")

    x = modelo[entradas].apply(pd.to_numeric, errors="raise").to_numpy(dtype=np.float64)
    exigir(np.isfinite(x).all() and (x >= 0).all(), "las entradas contienen faltantes, infinitos o negativos.")
    lluvia = pd.to_numeric(panel["precipitacion_mm"], errors="raise").astype(np.float64)
    susceptibilidad = pd.to_numeric(panel["susceptibilidad_mm_cenepred"], errors="raise").astype(np.float64)
    exigir(np.isfinite(lluvia).all() and (lluvia >= 0).all(), "lluvia de origen inválida.")
    exigir(np.isfinite(susceptibilidad).all() and susceptibilidad.gt(0).all(), "susceptibilidad inválida.")
    exigir(panel.groupby("ubigeo")["susceptibilidad_mm_cenepred"].nunique().eq(1).all(),
            "la referencia de susceptibilidad debe ser fija por distrito.")

    referencia = panel["periodo"].between("1981-01-01", "2002-12-01")
    exigir(int(referencia.sum()) == 3168, "la referencia del cuantil quedó incompleta.")
    q95_recalculado = float(np.quantile(lluvia[referencia], 0.95, method="linear"))
    q95 = float(parametros["q95_mm"])
    smax = float(parametros["susceptibilidad_max"])
    exigir(np.isclose(q95, 191.47574105, atol=1e-10, rtol=0), "Q95 configurado distinto al protocolo.")
    exigir(np.isclose(q95_recalculado, q95, atol=1e-8, rtol=0), "Q95 no se reproduce desde la referencia.")
    exigir(np.isclose(smax, 0.5003, atol=1e-12, rtol=0)
            and np.isclose(susceptibilidad.max(), smax, atol=1e-12, rtol=0), "Smax no coincide.")
    indice = (lluvia.to_numpy() / q95) * (susceptibilidad.to_numpy() / smax)
    indice_guardado = pd.to_numeric(panel["indice_proxy_estimado"], errors="raise").to_numpy(dtype=float)
    exigir(np.isfinite(indice_guardado).all()
            and np.allclose(indice, indice_guardado, atol=1e-6, rtol=0), "el índice guardado no se reproduce.")
    # side='right': en igualdad exacta 0.25→E2, 0.50→E3, 0.75→E4.
    # No se redondea el índice ni se recorta a uno para construir las etiquetas.
    etiquetas_recalculadas = np.asarray(bandas)[np.searchsorted(cortes, indice, side="right")]
    exigir(np.array_equal(etiquetas_recalculadas, panel[objetivo].to_numpy()), "las etiquetas no cumplen los cortes fijados.")

    panel["precipitacion_mm"] = lluvia
    grupos = panel.groupby("ubigeo", sort=False)["precipitacion_mm"]
    calculadas = pd.DataFrame({
        "precip_mes_previo_mm": grupos.shift(1),
        "precip_acum_3meses_previos_mm": grupos.transform(lambda s: s.shift(1).rolling(3, min_periods=3).sum()),
        "precip_acum_6meses_previos_mm": grupos.transform(lambda s: s.shift(1).rolling(6, min_periods=6).sum()),
    })
    for columna in calculadas:
        guardada = pd.to_numeric(panel[columna], errors="raise").to_numpy(dtype=float)
        exigir(np.allclose(guardada, calculadas[columna].to_numpy(), atol=1e-6, rtol=0, equal_nan=True),
                f"{columna} no coincide con los antecedentes de lluvia por distrito.")
    disponibles = calculadas.notna().all(axis=1)
    exigir(int((~disponibles).sum()) == 72, "deben excluirse exactamente los primeros seis meses por distrito.")
    exigir(panel.loc[~disponibles].groupby("ubigeo").size().eq(6).all()
            and panel.loc[~disponibles, "periodo"].le(pd.Timestamp("1981-06-01")).all(), "las exclusiones no son las primeras 72 filas.")
    exigir(np.array_equal(pd.to_numeric(panel["apto_modelo_proxy"], errors="raise").to_numpy(dtype=float),
                          disponibles.astype(int).to_numpy()), "la bandera de filas aptas no coincide.")
    esperadas_modelo = panel.loc[disponibles].sort_values(claves).reset_index(drop=True)
    exigir(modelo[claves].equals(esperadas_modelo[claves]), "modelado no es exactamente el subconjunto con seis antecedentes.")
    exigir(np.allclose(x, esperadas_modelo[entradas].to_numpy(dtype=float), atol=1e-6, rtol=0),
            "las cinco entradas de modelado difieren del panel completo.")
    exigir(np.array_equal(modelo[objetivo].to_numpy(), esperadas_modelo[objetivo].to_numpy()),
            "las etiquetas de modelado difieren del panel completo.")
    exigir(np.array_equal(modelo["distrito"].to_numpy(), esperadas_modelo["distrito"].to_numpy()),
            "los nombres de distrito de modelado difieren del panel completo.")

    particiones = [("entrenamiento", "1981-07-01", "2017-12-01", 5256),
                   ("validacion", "2018-01-01", "2021-12-01", 576),
                   ("prueba", "2022-01-01", "2025-12-01", 576)]
    exigir(config.get("filas_por_particion") == {nombre: n for nombre, _, _, n in particiones},
            "los tamaños de partición configurados cambiaron.")
    conteos = []
    for nombre, inicio, fin, n in particiones:
        mascara = modelo["periodo"].between(inicio, fin)
        exigir(int(mascara.sum()) == n and np.array_equal(mascara.to_numpy(), modelo["particion_temporal"].eq(nombre).to_numpy()),
                f"las filas y fechas de {nombre} no coinciden con el protocolo.")
        conteos.append({"Partición": nombre, "Inicio nominal": inicio[:7], "Fin nominal": fin[:7], "Filas": n})
    exigir(modelo.groupby("periodo")["particion_temporal"].nunique().eq(1).all(),
            "un mismo mes aparece en más de una partición.")
    exigir(np.array_equal(modelo["particion_temporal"].to_numpy(), esperadas_modelo["particion_temporal"].to_numpy()),
            "las particiones difieren entre ambos archivos.")

    exigir(geojson.get("type") == "FeatureCollection", "el mapa no es una colección GeoJSON.")
    geometria = geojson.get("features", [])
    exigir(len(geometria) == 12, "el mapa debe contener 12 geometrías distritales.")
    ids_geo = [f.get("properties", {}).get("ubigeo") for f in geometria]
    exigir(len(set(ids_geo)) == 12 and set(ids_geo) == ubigeos, "UBIGEO del mapa no coincide con el panel.")
    for f in geometria:
        g = f.get("geometry") or {}
        exigir(g.get("type") in {"Polygon", "MultiPolygon"}, "el mapa contiene una geometría no poligonal.")
        poligonos = [g.get("coordinates", [])] if g["type"] == "Polygon" else g.get("coordinates", [])
        exigir(len(poligonos) > 0, "hay geometrías vacías.")
        for poligono in poligonos:
            exigir(len(poligono) > 0, "hay polígonos sin anillos.")
            for anillo in poligono:
                coords = np.asarray(anillo, dtype=float)
                exigir(coords.ndim == 2 and coords.shape[0] >= 4 and coords.shape[1] == 2,
                        "anillo geográfico inválido.")
                exigir(np.isfinite(coords).all() and np.array_equal(coords[0], coords[-1]),
                        "hay anillos abiertos o coordenadas no finitas.")
                exigir((np.abs(coords[:, 0]) <= 180).all() and (np.abs(coords[:, 1]) <= 90).all(),
                        "las coordenadas del mapa no son longitud/latitud válidas.")

    return {"estado": "aprobada", "filas_completas": len(panel), "filas_modelado": len(modelo),
            "exclusiones_por_antecedentes": 72, "distritos": len(ubigeos), "q95_recalculado_mm": q95_recalculado,
            "susceptibilidad_max": smax, "particiones": conteos,
            "calendario": "nominal reconstruido; pendiente de confirmación nativa",
            "etiquetas": "E1–E4 calculadas; sin equivalencia con categorías oficiales"}


reporte_auditoria = validar_datos(df, completo, config, geojson, FEATURES, CLASES)
print("Auditoría aprobada: 6 480 filas de origen → 6 408 de modelado; 12 distritos; 72 exclusiones verificadas.")
print(f"Q95 reproducido: {reporte_auditoria['q95_recalculado_mm']:.8f} mm. Etiquetas, antecedentes y mapa coherentes.")
display(pd.DataFrame(reporte_auditoria["particiones"]))
print("Calendario nominal reconstruido; susceptibilidad fija sin vigencia histórica confirmada; bandas estimadas E1–E4.")


## 5 · Índice y etiquetas
La auditoría anterior ha recalculado el índice y los rezagos por distrito.

**I = (lluvia de t / 191,47574105) × (susceptibilidad / 0,5003)**

Q95 procede de 3.168 observaciones nominales de 1981–2002. Los cortes exploratorios
fijos son E1 < 0,25; E2 [0,25; 0,50); E3 [0,50; 0,75); E4 ≥ 0,75. No se redondea
antes de clasificar ni se limita I a uno. Estos cortes no proceden de una escala
oficial de peligro. Las etiquetas no son observaciones históricas del Mapa 17.


In [ ]:
diagnostico_cortes = []
referencia = completo['periodo'].between('1981-01-01','2002-12-01')
for q in [.90,.95,.99]:
    normalizador = np.quantile(completo.loc[referencia,'precipitacion_mm'], q, method='linear')
    indice = completo.precipitacion_mm / normalizador * completo.susceptibilidad_mm_cenepred / .5003
    alternativa = np.array(CLASES)[np.searchsorted([.25,.50,.75], indice, side='right')]
    diagnostico_cortes.append({'cuantil':q, 'normalizador_mm':normalizador,
        'filas_con_banda_distinta_a_Q95':int((alternativa != completo[TARGET]).sum())})
sensibilidad_indice = pd.DataFrame(diagnostico_cortes)
display(sensibilidad_indice)
print('Diagnóstico de la regla, no desempeño del clasificador. Se mantiene Q95 sin elegirlo por resultados de prueba.')


## 6 · Partición cronológica
Se conservan 5.256 filas de entrenamiento (1981-07–2017-12), 576 de validación
(2018–2021) y 576 de prueba (2022–2025). Los 12 distritos de un mes permanecen juntos.

La validación interna utiliza entrenamiento creciente y tres bloques completos:
2003–2007, 2008–2012 y 2013–2017. Todas sus evaluaciones son posteriores a la
referencia de Q95. Los ajustes, escalamiento y pesos usan únicamente su entrenamiento.


In [ ]:
train = df.loc[df.particion_temporal.eq('entrenamiento')].copy().reset_index(drop=True)
val = df.loc[df.particion_temporal.eq('validacion')].copy().reset_index(drop=True)
test = df.loc[df.particion_temporal.eq('prueba')].copy().reset_index(drop=True)
assert (len(train),len(val),len(test)) == (5256,576,576)
assert train.periodo.max() < val.periodo.min() <= val.periodo.max() < test.periodo.min()
FOLDS = []
filas_cortes = []
for inicio, fin in [('2003-01-01','2007-12-01'),('2008-01-01','2012-12-01'),('2013-01-01','2017-12-01')]:
    ia = np.flatnonzero(train.periodo.lt(pd.Timestamp(inicio)))
    ie = np.flatnonzero(train.periodo.between(inicio,fin))
    assert train.iloc[ia].periodo.max() < train.iloc[ie].periodo.min()
    assert train.iloc[ie].groupby('periodo').size().eq(12).all()
    FOLDS.append((ia,ie))
    filas_cortes.append({'ajuste_hasta':str(train.iloc[ia].periodo.max().date()),
                         'evaluacion_desde':inicio[:7], 'evaluacion_hasta':fin[:7],
                         'filas_ajuste':len(ia), 'filas_evaluacion':len(ie)})
display(pd.DataFrame(filas_cortes))


## 7 · Exploración de entrenamiento
Los gráficos exploratorios usan sólo entrenamiento. Las distribuciones de prueba
no se utilizan para decidir variables, parámetros ni cortes.


In [ ]:
fig, ejes = plt.subplots(1,2,figsize=(12,4))
train[TARGET].value_counts().reindex(CLASES,fill_value=0).plot.bar(ax=ejes[0],color=['#2e7d32','#fbc02d','#ef6c00','#c62828'])
ejes[0].set(title='Bandas de referencia · entrenamiento',xlabel='Banda calculada',ylabel='Filas')
train.groupby('mes').precip_mes_previo_mm.mean().plot(marker='o',ax=ejes[1])
ejes[1].set(title='Lluvia previa media · entrenamiento',xlabel='Mes objetivo',ylabel='mm')
fig.tight_layout()
fig.savefig(OUT / 'exploracion_entrenamiento.png',dpi=180,bbox_inches='tight')
plt.show()
display(train.groupby('distrito')[FEATURES[2:]].mean().round(2))
fig, ax = plt.subplots(figsize=(8,5))
sns.heatmap(train[FEATURES].corr(),annot=True,fmt='.2f',cmap='coolwarm',vmin=-1,vmax=1,ax=ax)
ax.set_title('Correlación entre entradas · entrenamiento')
fig.tight_layout()
fig.savefig(OUT / 'correlaciones_entrenamiento.png',dpi=180,bbox_inches='tight')
plt.show()


## 8 · Pipelines y desbalance
El Árbol de Decisión sirve de referencia. Los cuatro algoritmos reciben las mismas
cinco entradas. SVM incluye escalamiento dentro de cada entrenamiento; los árboles
no lo necesitan. Los pesos equilibrados se recalculan dentro de cada ajuste.
No se generan etiquetas, observaciones sintéticas ni imputaciones de antecedentes.


## 9 · Comparación y selección
Se prueban cuatro configuraciones por algoritmo y tres cortes por configuración
(48 ajustes internos). Para cada algoritmo se escoge su configuración por F1-macro
medio de CV. Después se ajusta con entrenamiento completo y se compara en validación.
El ganador se selecciona por F1-macro de validación, luego exactitud balanceada y,
en empate exacto, la prioridad Árbol → Forest → XGBoost → SVM.
Se conserva su ajuste con entrenamiento; no se reentrena usando validación.
Esta celda puede tardar varios minutos en CPU y muestra el avance por algoritmo.


In [ ]:
"""Comparación temporal de cuatro clasificadores; esta celda no consulta prueba."""
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

CODIGOS = np.arange(len(CLASES))
codificacion = {banda: codigo for codigo, banda in enumerate(CLASES)}


def codificar_etiquetas(serie):
    codigos = serie.map(codificacion)
    if codigos.isna().any():
        raise ValueError('Se encontraron etiquetas fuera de E1–E4.')
    return codigos.to_numpy(dtype=int)


X_train = train[FEATURES].copy()
y_train = codificar_etiquetas(train[TARGET])
X_val = val[FEATURES].copy()
y_val = codificar_etiquetas(val[TARGET])
assert set(y_train) == set(CODIGOS), 'Entrenamiento debe contener las cuatro bandas.'

plantillas = {
    'Árbol de Decisión': Pipeline([
        ('escala', 'passthrough'),
        ('clasificador', DecisionTreeClassifier(random_state=SEED)),
    ]),
    'Random Forest': Pipeline([
        ('escala', 'passthrough'),
        ('clasificador', RandomForestClassifier(random_state=SEED, n_jobs=2)),
    ]),
    'XGBoost': Pipeline([
        ('escala', 'passthrough'),
        ('clasificador', XGBClassifier(
            objective='multi:softprob', num_class=4, eval_metric='mlogloss',
            tree_method='hist', random_state=SEED, n_jobs=2,
        )),
    ]),
    'SVM': Pipeline([
        ('escala', StandardScaler()),
        ('clasificador', SVC(kernel='rbf', probability=False, cache_size=256)),
    ]),
}
# Cuatro configuraciones fijadas por algoritmo antes de consultar validación.
opciones = {
    'Árbol de Decisión': [
        {'max_depth': 3, 'min_samples_leaf': 10},
        {'max_depth': 5, 'min_samples_leaf': 10},
        {'max_depth': 8, 'min_samples_leaf': 5},
        {'max_depth': None, 'min_samples_leaf': 10},
    ],
    'Random Forest': [
        {'n_estimators': 200, 'max_depth': 6, 'min_samples_leaf': 3},
        {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 5},
        {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 3},
        {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 8},
    ],
    'XGBoost': [
        {'n_estimators': 100, 'max_depth': 2, 'learning_rate': .05, 'reg_lambda': 2, 'min_child_weight': 5},
        {'n_estimators': 200, 'max_depth': 2, 'learning_rate': .05, 'reg_lambda': 2, 'min_child_weight': 5},
        {'n_estimators': 100, 'max_depth': 3, 'learning_rate': .1, 'reg_lambda': 2, 'min_child_weight': 5},
        {'n_estimators': 200, 'max_depth': 3, 'learning_rate': .05, 'reg_lambda': 2, 'min_child_weight': 5},
    ],
    'SVM': [
        {'C': .5, 'gamma': 'scale'}, {'C': 2, 'gamma': 'scale'},
        {'C': 8, 'gamma': 'scale'}, {'C': 2, 'gamma': .1},
    ],
}


def ajustar_con_pesos(pipeline, X, y):
    # Los pesos se calculan de nuevo SOLO con las etiquetas de este entrenamiento.
    pesos = compute_sample_weight(class_weight='balanced', y=y)
    return pipeline.fit(X, y, clasificador__sample_weight=pesos)


filas_cv, filas_pliegues, filas_validacion = [], [], []
mejores_modelos, mejores_parametros = {}, {}
for nombre, plantilla in plantillas.items():
    print(f'Comparando {nombre}: {len(opciones[nombre])} configuraciones × {len(FOLDS)} cortes...')
    candidatos = []
    for numero, parametros in enumerate(opciones[nombre], 1):
        ajustes = {f'clasificador__{clave}': valor for clave, valor in parametros.items()}
        puntuaciones = []
        inicio = time.perf_counter()
        for pliegue, (indices_ajuste, indices_evaluacion) in enumerate(FOLDS, 1):
            ia, ie = np.asarray(indices_ajuste), np.asarray(indices_evaluacion)
            assert len(ia) and len(ie) and not np.intersect1d(ia, ie).size
            assert train.iloc[ia]['periodo'].max() < train.iloc[ie]['periodo'].min()
            assert train.iloc[ie]['periodo'].min() >= pd.Timestamp('2003-01-01')
            assert set(y_train[ia]) == set(CODIGOS), 'Faltan bandas en un entrenamiento interno.'
            modelo = clone(plantilla).set_params(**ajustes)
            ajustar_con_pesos(modelo, X_train.iloc[ia], y_train[ia])
            pred = modelo.predict(X_train.iloc[ie])
            score = float(f1_score(y_train[ie], pred, labels=CODIGOS, average='macro', zero_division=0))
            puntuaciones.append(score)
            filas_pliegues.append({
                'modelo': nombre, 'configuracion': numero, 'pliegue': pliegue,
                'inicio_evaluacion': str(train.iloc[ie]['periodo'].min().date()),
                'fin_evaluacion': str(train.iloc[ie]['periodo'].max().date()),
                'f1_macro': score,
            })
        registro = {
            'modelo': nombre, 'configuracion': numero,
            'parametros': json.dumps(parametros, ensure_ascii=False, sort_keys=True),
            'f1_macro_cv': float(np.mean(puntuaciones)),
            'desviacion_cv': float(np.std(puntuaciones)),
            'segundos_cv': time.perf_counter() - inicio,
        }
        filas_cv.append(registro)
        candidatos.append((registro, ajustes, parametros))
    elegido = sorted(candidatos, key=lambda x: (-x[0]['f1_macro_cv'], x[0]['desviacion_cv'], x[0]['configuracion']))[0]
    registro, ajustes, parametros = elegido
    modelo = clone(plantilla).set_params(**ajustes)
    inicio = time.perf_counter()
    ajustar_con_pesos(modelo, X_train, y_train)
    pred_val = modelo.predict(X_val)
    mejores_modelos[nombre] = modelo
    mejores_parametros[nombre] = parametros
    filas_validacion.append({
        'modelo': nombre, 'f1_macro_cv': registro['f1_macro_cv'],
        'f1_macro_validacion': float(f1_score(y_val, pred_val, labels=CODIGOS, average='macro', zero_division=0)),
        'exactitud_balanceada_validacion': float(balanced_accuracy_score(y_val, pred_val)),
        'segundos_ajuste_final': time.perf_counter() - inicio,
        'parametros': registro['parametros'],
    })

resultados_cv = pd.DataFrame(filas_cv)
resultados_pliegues = pd.DataFrame(filas_pliegues)
prioridad = {nombre: i for i, nombre in enumerate(plantillas)}
resumen_validacion = pd.DataFrame(filas_validacion)
resumen_validacion['_prioridad'] = resumen_validacion['modelo'].map(prioridad)
resumen_validacion = resumen_validacion.sort_values(
    ['f1_macro_validacion', 'exactitud_balanceada_validacion', '_prioridad'],
    ascending=[False, False, True], kind='stable',
).drop(columns='_prioridad').reset_index(drop=True)
nombre_final = resumen_validacion.iloc[0]['modelo']
modelo_final = mejores_modelos[nombre_final]
print('Modelo seleccionado mediante validación:', nombre_final)
print('Se conserva el ajuste realizado exclusivamente con entrenamiento; no se reentrena con validación.')
display(resumen_validacion)


## 10 · Evaluación reservada e interpretación
Se evalúa únicamente el algoritmo seleccionado sobre prueba. Se reutilizan sus
predicciones en el mapa. Se informan F1-macro, exactitud balanceada, precisión y
sensibilidad por banda y matriz de confusión. Las metas F1-macro ≥ 0,70 y recall
E4 ≥ 0,75 se comprueban, sin asumir que se alcanzan.

La AUC OVR se calcula sólo si existen probabilidades compatibles; SVM no usa
calibración aleatoria y puede omitirla. La importancia por permutación se calcula
en validación y no representa causalidad. No se cambian modelos por el resultado de prueba.


In [ ]:
"""Evaluación reservada del modelo seleccionado y explicación con validación."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, recall_score, roc_auc_score, make_scorer,
)
from sklearn.inspection import permutation_importance

# Esta es la única predicción de prueba del experimento; se reutiliza en el mapa.
y_test = codificar_etiquetas(test[TARGET])
pred_test_codigos = np.asarray(modelo_final.predict(test[FEATURES]), dtype=int)
assert set(pred_test_codigos).issubset(set(CODIGOS))
test = test.copy()
test['prediccion'] = np.asarray(CLASES)[pred_test_codigos]
testpred = test[['ubigeo', 'distrito', 'periodo', TARGET, 'prediccion']].copy()
sensibilidades = recall_score(y_test, pred_test_codigos, labels=CODIGOS, average=None, zero_division=0)
metricas = {
    'modelo': nombre_final, 'filas_prueba': int(len(test)),
    'f1_macro': float(f1_score(y_test, pred_test_codigos, labels=CODIGOS, average='macro', zero_division=0)),
    'exactitud_balanceada': float(balanced_accuracy_score(y_test, pred_test_codigos)),
    'exactitud': float(accuracy_score(y_test, pred_test_codigos)),
    'recall_E4': float(sensibilidades[3]),
}
metricas['meta_f1_macro_070'] = metricas['f1_macro'] >= .70
metricas['meta_recall_E4_075'] = metricas['recall_E4'] >= .75
reporte = classification_report(
    y_test, pred_test_codigos, labels=CODIGOS, target_names=CLASES,
    output_dict=True, zero_division=0,
)
matriz = confusion_matrix(y_test, pred_test_codigos, labels=CODIGOS)
auc_motivo = 'Omitida: el modelo no dispone de probabilidades multiclase compatibles.'
if hasattr(modelo_final, 'predict_proba') and set(y_test) == set(CODIGOS):
    clases_modelo = np.asarray(modelo_final.classes_)
    if np.array_equal(clases_modelo, CODIGOS):
        probabilidades = np.asarray(modelo_final.predict_proba(test[FEATURES]))
        if (probabilidades.shape == (len(test), 4) and np.isfinite(probabilidades).all()
                and (probabilidades >= 0).all()
                and np.allclose(probabilidades.sum(axis=1), 1, atol=1e-5)):
            metricas['auc_ovr_macro'] = float(roc_auc_score(
                y_test, probabilidades, labels=CODIGOS, multi_class='ovr', average='macro',
            ))
            auc_motivo = 'AUC OVR macro calculada con probabilidades de banda; no son probabilidades de desastre.'
metricas['nota_auc'] = auc_motivo
print('Resultados finales sobre 2022–2025. Las referencias son bandas calculadas del índice.')
display(pd.DataFrame([metricas]))
display(pd.DataFrame(reporte).T)

OUT.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 5))
imagen = ax.imshow(matriz, cmap='Blues')
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(matriz[i, j]), ha='center', va='center',
                color='white' if matriz[i, j] > matriz.max() / 2 else 'black')
ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASES, yticklabels=CLASES,
       xlabel='Predicción del modelo', ylabel='Referencia calculada',
       title=f'Matriz de confusión · {nombre_final}')
fig.colorbar(imagen, ax=ax)
fig.tight_layout()
fig.savefig(OUT / 'matriz_confusion.png', dpi=180, bbox_inches='tight')
plt.show()

# La importancia usa validación, manteniendo prueba fuera de este diagnóstico.
scorer_importancia = make_scorer(f1_score, labels=CODIGOS, average='macro', zero_division=0)
permutacion = permutation_importance(
    modelo_final, X_val, y_val, scoring=scorer_importancia,
    n_repeats=10, random_state=SEED, n_jobs=2,
)
importancias = pd.DataFrame({
    'variable': FEATURES, 'caida_f1_macro_media': permutacion.importances_mean,
    'desviacion': permutacion.importances_std,
}).sort_values('caida_f1_macro_media', ascending=False).reset_index(drop=True)
print('Importancia por permutación sobre validación: contribución predictiva, no causalidad física.')
display(importancias)
fig, ax = plt.subplots(figsize=(8, 4))
orden = importancias.iloc[::-1]
ax.barh(orden['variable'], orden['caida_f1_macro_media'], xerr=orden['desviacion'])
ax.axvline(0, color='black', linewidth=.8)
ax.set(xlabel='Caída de F1-macro al permutar', title='Importancia en validación')
fig.tight_layout()
fig.savefig(OUT / 'importancias_validacion.png', dpi=180, bbox_inches='tight')
plt.show()


## 11 · Interfaz y mapa distrital
**Histórico:** predicciones de los 12 distritos para un mes de prueba y detalle
opcional de distrito, con la referencia calculada identificada aparte.

**Manual:** seis lluvias previas al mes objetivo. La susceptibilidad del distrito
es fija y de sólo lectura; únicamente se colorea el distrito consultado.
El mapa muestra bandas E1–E4 a escala distrital, sin interpolar zonas interiores.
Necesita conexión para cargar el mapa base y una sesión activa para los controles.


In [ ]:
"""Interfaz académica del índice E1–E4, para ejecutar dentro de Colab."""
import copy
import html
import numpy as np
import pandas as pd
import folium
import ipywidgets as widgets
from IPython.display import display, HTML

COLORES = dict(zip(CLASES, ['#2e7d32', '#fbc02d', '#ef6c00', '#c62828']))
referencia_distrital = df[['ubigeo', 'distrito', 'susceptibilidad_mm_cenepred']].drop_duplicates()
assert not referencia_distrital['ubigeo'].duplicated().any(), 'Susceptibilidad no única por distrito.'
referencia_distrital = referencia_distrital.set_index('ubigeo').sort_index()
assert {str(f['properties']['ubigeo']) for f in geojson['features']} == set(referencia_distrital.index)


def _banda(valor):
    if str(valor) in CLASES:
        return str(valor)
    if isinstance(valor, (int, np.integer)) and 0 <= valor < len(CLASES):
        return CLASES[int(valor)]
    raise ValueError('La predicción no corresponde a E1, E2, E3 o E4.')


def crear_mapa(valores, detalle=None):
    """Colorea exclusivamente UBIGEO con resultado; los restantes quedan grises."""
    capa = copy.deepcopy(geojson)
    for feature in capa['features']:
        originales = feature['properties']
        codigo = str(originales['ubigeo'])
        feature['properties'] = {
            'ubigeo': codigo, 'distrito': originales['nombdist'],
            'resultado': _banda(valores[codigo]) if codigo in valores else 'Sin resultado en este escenario',
            'detalle': str((detalle or {}).get(codigo, '')),
        }
    mapa = folium.Map(location=[-7.15, -78.45], zoom_start=9, tiles='OpenStreetMap')
    layer = folium.GeoJson(
        capa, name='Bandas estimadas por distrito',
        style_function=lambda f: {
            'fillColor': COLORES.get(f['properties']['resultado'], '#bdbdbd'),
            'color': '#333333', 'weight': 1, 'fillOpacity': .70,
        },
        highlight_function=lambda f: {'weight': 3, 'fillOpacity': .85},
        tooltip=folium.GeoJsonTooltip(
            fields=['distrito', 'ubigeo', 'resultado', 'detalle'],
            aliases=['Distrito', 'UBIGEO', 'Predicción', 'Referencia / contexto'], sticky=False,
        ),
    ).add_to(mapa)
    mapa.fit_bounds(layer.get_bounds())
    leyenda = ' &nbsp; '.join(
        f'<span style="color:{color}">■</span> {banda}' for banda, color in COLORES.items()
    )
    mapa.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:20px;left:20px;z-index:9999;background:white;'
        'padding:12px;border:1px solid #777;font-size:13px">'
        '<b>Bandas estimadas del índice</b><br>' + leyenda +
        '<br><span style="color:#bdbdbd">■</span> Sin resultado'
        '<br>No son niveles oficiales de riesgo.</div>'
    ))
    return mapa


opciones_distritos = [(fila.distrito, codigo) for codigo, fila in referencia_distrital.iterrows()]
periodos_disponibles = sorted(pd.to_datetime(test['periodo']).dt.strftime('%Y-%m').unique())
mes_historico = widgets.Dropdown(options=periodos_disponibles, description='Mes:')
distrito_historico = widgets.Dropdown(options=[('Todos', '')] + opciones_distritos, description='Distrito:')
boton_historico = widgets.Button(description='Consultar histórico', button_style='primary')
salida_historico = widgets.Output()


def consultar_historico(_=None):
    with salida_historico:
        salida_historico.clear_output(wait=True)
        try:
            filas = test.loc[pd.to_datetime(test['periodo']).dt.strftime('%Y-%m').eq(mes_historico.value)].copy()
            if filas['ubigeo'].duplicated().any():
                raise ValueError('Hay más de un resultado por distrito en el mes elegido.')
            valores = {str(r.ubigeo): _banda(r.prediccion) for r in filas.itertuples()}
            detalle = {str(r.ubigeo): 'Referencia calculada: ' + str(r.banda_proxy_estimada) for r in filas.itertuples()}
            display(HTML('<b>Predicciones del periodo de prueba</b> · Referencias calculadas con la regla del índice.'))
            tabla = filas if not distrito_historico.value else filas.loc[filas.ubigeo.eq(distrito_historico.value)]
            display(tabla[['distrito', 'periodo', 'prediccion', 'banda_proxy_estimada']].rename(columns={
                'prediccion': 'Predicción del modelo', 'banda_proxy_estimada': 'Referencia calculada',
            }))
            display(crear_mapa(valores, detalle))
        except (ValueError, KeyError, TypeError) as error:
            display(HTML('<b>No se pudo consultar:</b> ' + html.escape(str(error))))


boton_historico.on_click(consultar_historico)
distrito_manual = widgets.Dropdown(options=opciones_distritos, description='Distrito:')
anio_manual = widgets.BoundedIntText(value=2026, min=1982, max=2100, description='Año objetivo:')
mes_manual = widgets.Dropdown(options=[(str(i).zfill(2), i) for i in range(1, 13)], description='Mes objetivo:')
susceptibilidad_visible = widgets.HTML()
lluvias = [widgets.BoundedFloatText(value=0, min=0, max=1e9, step=1, description=f't−{i}:') for i in range(1, 7)]
boton_manual = widgets.Button(description='Predecir banda', button_style='primary')
salida_manual = widgets.Output()


def actualizar_antecedentes(_=None):
    objetivo = pd.Timestamp(year=anio_manual.value, month=mes_manual.value, day=1)
    for i, campo in enumerate(lluvias, 1):
        campo.description = (objetivo - pd.DateOffset(months=i)).strftime('%Y-%m') + ' mm:'
        campo.style.description_width = '110px'
    valor_s = referencia_distrital.loc[distrito_manual.value, 'susceptibilidad_mm_cenepred']
    susceptibilidad_visible.value = f'<b>Susceptibilidad fija del distrito:</b> {valor_s:.4f} (solo lectura)'


def predecir_manual(_):
    with salida_manual:
        salida_manual.clear_output(wait=True)
        try:
            antecedentes = np.array([campo.value for campo in lluvias], dtype=float)
            if not np.isfinite(antecedentes).all() or (antecedentes < 0).any():
                raise ValueError('Ingrese seis precipitaciones finitas, no negativas y expresadas en mm.')
            codigo = distrito_manual.value
            fila = pd.DataFrame([{
                'mes': mes_manual.value,
                'susceptibilidad_mm_cenepred': referencia_distrital.loc[codigo, 'susceptibilidad_mm_cenepred'],
                'precip_mes_previo_mm': antecedentes[0],
                'precip_acum_3meses_previos_mm': antecedentes[:3].sum(),
                'precip_acum_6meses_previos_mm': antecedentes.sum(),
            }], columns=FEATURES)
            banda = _banda(modelo_final.predict(fila)[0])
            display(HTML(f'<b>Predicción: {banda}</b> · {anio_manual.value}-{mes_manual.value:02d}'))
            display(fila)
            display(HTML('Escenario ingresado por el usuario. La lluvia del mes objetivo y su etiqueta de referencia no se conocen.'))
            display(crear_mapa({codigo: banda}, {codigo: 'Escenario manual; sin referencia observada'}))
        except (ValueError, KeyError, TypeError, OverflowError) as error:
            display(HTML('<b>Revise los datos:</b> ' + html.escape(str(error))))


def invalidar_manual(_):
    salida_manual.clear_output(wait=False)


for control in [distrito_manual, anio_manual, mes_manual] + lluvias:
    control.observe(invalidar_manual, names='value')
for control in (mes_historico, distrito_historico):
    control.observe(lambda _: salida_historico.clear_output(wait=False), names='value')
for control in (distrito_manual, anio_manual, mes_manual):
    control.observe(actualizar_antecedentes, names='value')
boton_manual.on_click(predecir_manual)
actualizar_antecedentes()
interfaz = widgets.Tab(children=[
    widgets.VBox([widgets.HBox([mes_historico, distrito_historico]), boton_historico, salida_historico]),
    widgets.VBox([widgets.HBox([distrito_manual, anio_manual, mes_manual]), susceptibilidad_visible,
                  widgets.HTML('<b>Seis lluvias previas al mes objetivo:</b>'),
                  widgets.GridBox(lluvias, layout=widgets.Layout(grid_template_columns='repeat(2, 330px)')),
                  boton_manual, salida_manual]),
])
interfaz.set_title(0, 'Histórico de prueba')
interfaz.set_title(1, 'Escenario manual')
display(HTML('<h3>Consulta del índice estimado E1–E4</h3><p>Referencia territorial fija; fechas nominales reconstruidas. Resolución distrital.</p>'))
display(interfaz)
consultar_historico()


## 12 · Exportación y evidencias
Se guardan modelo, configuración, versiones, huellas de entrada, auditoría,
comparaciones, métricas y predicciones reales de esta ejecución. Los mapas HTML
se abren fuera de Colab; los controles de consulta manual necesitan ejecutar Python.
El ZIP de resultados no modifica los archivos de origen.


In [ ]:
import joblib, zipfile
joblib.dump(modelo_final,OUT / 'modelo_final.joblib')
resultados_cv.to_csv(OUT / 'comparacion_cv.csv',index=False,encoding='utf-8-sig')
resultados_pliegues.to_csv(OUT / 'resultados_pliegues.csv',index=False,encoding='utf-8-sig')
resumen_validacion.to_csv(OUT / 'comparacion_validacion.csv',index=False,encoding='utf-8-sig')
testpred.assign(periodo=testpred.periodo.dt.strftime('%Y-%m')).to_csv(OUT / 'predicciones_prueba.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(reporte).T.to_csv(OUT / 'metricas_por_banda.csv',encoding='utf-8-sig')
pd.DataFrame(matriz,index=CLASES,columns=CLASES).to_csv(OUT / 'matriz_confusion.csv',encoding='utf-8-sig')
importancias.to_csv(OUT / 'importancias_validacion.csv',index=False,encoding='utf-8-sig')
sensibilidad_indice.to_csv(OUT / 'sensibilidad_regla.csv',index=False,encoding='utf-8-sig')
for nombre,objeto in [('metricas_prueba.json',metricas),('auditoria.json',reporte_auditoria),
                      ('versiones.json',VERSIONES),('huellas_fuentes.json',ESPERADOS)]:
    (OUT / nombre).write_text(json.dumps(objeto,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
protocolo = {'semilla':SEED,'X':FEATURES,'objetivo':TARGET,'clases':CLASES,
             'modelo_elegido':nombre_final,'parametros':mejores_parametros[nombre_final],
             'particiones':config['filas_por_particion'],'cv':filas_cortes,
             'ajuste_final':'solo entrenamiento 1981-07 a 2017-12; sin refit de validacion',
             'seleccion':'CV dentro de train, luego F1-macro de validacion para elegir algoritmo',
             'regla':config['parametros'],'limitaciones':config['advertencias']}
(OUT / 'protocolo.json').write_text(json.dumps(protocolo,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT / 'requirements_ejecucion.txt').write_text('\n'.join(f'{p}=={v}' for p,v in VERSIONES.items())+'\n',encoding='utf-8')
for periodo, filas in test.groupby('periodo'):
    valores = dict(zip(filas.ubigeo,filas.prediccion))
    detalle = dict(zip(filas.ubigeo,'Referencia calculada: '+filas[TARGET]))
    crear_mapa(valores,detalle).save(str(OUT / ('mapa_'+periodo.strftime('%Y-%m')+'.html')))
manual = 'Resultados de un índice exploratorio E1–E4, no validación de peligro físico observado.\nEl modelo utiliza cinco entradas; para cargarlo se necesitan las versiones registradas.\nUse joblib.load sólo con el archivo generado por usted y de procedencia confiable.\nLos 48 mapas muestran predicciones almacenadas; la consulta manual funciona con el cuaderno activo.\nConsulte protocolo.json y auditoria.json para el alcance y las limitaciones.\n'
(OUT / 'LEEME_RESULTADOS.txt').write_text(manual,encoding='utf-8')
archivo_resultados = BASE / 'resultados_cajamarca.zip'
with zipfile.ZipFile(archivo_resultados,'w',zipfile.ZIP_DEFLATED) as archivo:
    for fichero in sorted(OUT.iterdir()):
        if fichero.is_file():
            archivo.write(fichero,fichero.name)
print('Resultados guardados en:',archivo_resultados)
if EN_COLAB:
    from google.colab import files
    files.download(str(archivo_resultados))


### Referencias técnicas
- [Pipelines de scikit-learn](https://scikit-learn.org/stable/modules/compose.html)
- [Validación temporal](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [API de XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html)
- [Widgets de Colab](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb)

Los cortes mensuales se construyen explícitamente para mantener juntos los 12 distritos;
no se aplica TimeSeriesSplit directamente a las filas del panel.
Las fuentes y sus incertidumbres están documentadas en LEEME_METODO.md dentro del paquete.
